# Visual Data Science Notebook on Renewable Energy Insight
This notebook helps to work through the pipeline of getting interesting information and metrics on renewable energies

## Return on investment of different countries

In [ ]:
import sys
print(sys.executable)
import pandas as pd
import numpy as np
import chardet
import seaborn as snus
import matplotlib.pyplot as plt
from enum import Enum   
DATA_FOLDER = "data/"
SELECTED_COUNTRIES = ["Total IEA","Germany","Mexico", "Brazil","Austria", "France", "Italy", "Spain", "United Kingdom", "Netherlands", "Sweden"]

class RDD_TYPE(Enum):
    TOTAL = "TOTAL"
    RENEWABLE = "RENEWABLE"
    FOSSILFUEL = "FOSSILFUEL"

In [188]:
# with open(DATA_FOLDER + 'capacity-generation-emission.csv', 'rb') as f:
#     result = chardet.detect(f.read())
#     encoding = result['encoding']
capacity_generation_emission_content = pd.read_csv(DATA_FOLDER + "capacity-generation-emission.csv", encoding='utf-8')
#print(capacity_generation_emission_content.head())

# with open(DATA_FOLDER + 'public-investments.csv', 'rb') as f:
#     result = chardet.detect(f.read())
#     encoding = result['encoding']
# print(f"Detected encoding: {encoding}")
public_investment_content = pd.read_csv(DATA_FOLDER + 'public-investments.csv', encoding='iso-8859-1')

# with open(DATA_FOLDER + 'european-energy-prices.csv', 'rb') as f:
#     result = chardet.detect(f.read())
#     encoding = result['encoding']
european_energy_prices_content = pd.read_csv(DATA_FOLDER + 'european-energy-prices.csv', encoding='iso-8859-1')
#print(european_energy_prices_content.head())

gdp_population_content = pd.read_csv(DATA_FOLDER + "gdp-population.csv")

rdd_content = pd.read_csv(DATA_FOLDER + "rdd-total-fossile-renewable.csv")








In [ ]:
rdd_content = rdd_content.drop(
    columns=[
        "STRUCTURE", "STRUCTURE_ID", "STRUCTURE_NAME", "ACTION", "COUNTRY", "FREQUENCY", "Frequency",
        "RDD_SECTOR","Technology", "RDD_TYPE","Sector", "Type", "UNIT","Time Period","Observation value", "QUALIFIER", "Qualifier",
        "CONF_STATUS", "Confidential Status", "Decimals","Unit multiplier","DECIMALS"
    ]
)

#Remove unneeded columns from gdp_population_content
gdp_population_content = gdp_population_content.drop(
    columns=[   
             "Country Code", "Series Code"
    ]
)
#Remove [YRXXXX] from column names
gdp_population_content.columns = gdp_population_content.columns.str.replace(r'\[YR\d{4}\]', '', regex=True).str.strip()

Index(['Country Name', 'Series Name', '1960', '1961', '1962', '1963', '1964',
       '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973',
       '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982',
       '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991',
       '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000',
       '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009',
       '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018',
       '2019', '2020', '2021', '2022', '2023', '2024'],
      dtype='object')


In [ ]:
def get_average_european_energy_price(country_list):
    country_list_df = pd.DataFrame()
    for country in country_list:
        country_data = european_energy_prices_content[european_energy_prices_content['Country'] == country].copy()
        country_data['Year'] = pd.to_datetime(country_data['Date']).dt.year
        country_data['Avarage Price'] = country_data.groupby(['Year'])['Price (EUR/MWhe)'].transform('mean')
        country_data = country_data.drop(['Date', 'Price (EUR/MWhe)'], axis=1)
        country_data = country_data.drop_duplicates()
        country_list_df = pd.concat([country_list_df, country_data], ignore_index=True)
       

    return country_list_df

In [ ]:
country_prices = [get_average_european_energy_price(country) for country in [["Austria","Germany"]]]
print(country_prices)

In [ ]:
def get_yearly_public_investment(country_list):
    country_list_df = pd.DataFrame()
    for country in country_list:
        country_data = public_investment_content[public_investment_content['Country/area'] == country].copy()
        country_list_df = pd.concat([country_list_df, country_data], ignore_index=True)
    return country_list_df

def get_countries_with_most_data(year=None, top_n=10, technology='Multiple renewables'):
    # Count countries that have a non-missing / non-'-' value in the Public Investments column.
    col = 'Public Investments (2022 Million USD)'
    df = public_investment_content.copy()

    # optional year filter if the dataset has a Year column
    if year is not None and 'Year' in df.columns:
        df = df[df['Year'] == year]
    # consider non-null and not the hyphen placeholder
    valid = df[df[col].notna() & (df[col] != '-')]

    # count valid rows per country and return top N countries
    top_countries = valid['Country/area'].value_counts().head(top_n).index.tolist()
    num_valid_rows = valid['Country/area'].value_counts().head(top_n).tolist()
    top_countries = (top_countries, num_valid_rows)
    return top_countries

def get_investment_by_country_and_year(country_list, technology='Multiple renewables'):
    country_list_df = pd.DataFrame()
    for country in country_list:
        country_data = public_investment_content[public_investment_content['Country/area'] == country].copy()
        country_data = country_data[country_data['Technology'] == technology]
        country_list_df = pd.concat([country_list_df, country_data], ignore_index=True)
    country_list_df.rename(columns={'Public Investments (2022 Million USD)': 'Public Investments (Mill USD)',"Country/area": "Country"}, inplace=True)
    country_list_df.drop(['Technology'], axis=1, inplace=True)
    return country_list_df

In [ ]:
investments = get_yearly_public_investment(SELECTED_COUNTRIES)
# print(investments)

top_countries = get_countries_with_most_data(top_n=50)
print(top_countries)

investment = get_investment_by_country_and_year(SELECTED_COUNTRIES) 
# yearly_investment = investment.pivot(index='Year', columns='Country', values='Public Investments (Mill USD)')
# print(yearly_investment)
investment['Public Investments (Mill USD)'] = pd.to_numeric(
    investment['Public Investments (Mill USD)'], 
    errors='coerce'  # Converts '-' to NaN
)
investment_clean = investment.dropna(subset=['Public Investments (Mill USD)'])
pd.set_option('display.max_rows', None)
print(investment_clean)

In [ ]:
snus.set_theme(style="whitegrid")
snus.lineplot(
    data=investment_clean, 
    x='Year', 
    y='Public Investments (Mill USD)', 
    hue='Country', 
    marker='o'
)
plt.title('Investment Over Time by Country')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
def get_rdd_investment(country_list, type: RDD_TYPE):
    country_list_df = pd.DataFrame()
    for country in country_list:
        country_data = rdd_content[rdd_content['Country/Region']==country]
        rdd_type = country_data[country_data['RDD_TECH']==type.value]
        country_list_df = pd.concat([country_list_df, rdd_type], ignore_index=True)
    return country_list_df

In [ ]:
selected_renewable_rdd = get_rdd_investment(SELECTED_COUNTRIES, RDD_TYPE.RENEWABLE)
selected_fossilfuel_rdd = get_rdd_investment(SELECTED_COUNTRIES, RDD_TYPE.FOSSILFUEL)
selected_total_rdd = get_rdd_investment(SELECTED_COUNTRIES, RDD_TYPE.TOTAL)
selected_renewable_rdd['Investment_Type'] = 'Renewable'
selected_fossilfuel_rdd['Investment_Type'] = 'Fossil Fuel'
selected_total_rdd['Investment_Type'] = 'Total'
combined_rdd = pd.concat([selected_renewable_rdd, selected_fossilfuel_rdd, selected_total_rdd], ignore_index=True)

#Plot Renewable vs Fossilfuel RDD Investment Over Time by Country

for country in SELECTED_COUNTRIES:
    country_data = combined_rdd[combined_rdd['Country/Region'] == country]
    
    plt.figure(figsize=(10, 6))
    snus.lineplot(
        data=country_data,
        x='TIME_PERIOD',
        y='OBS_VALUE',
        hue='Investment_Type',
        marker='o',
        linewidth=2
    )
    plt.title(f'Renewable vs Fossil Fuel RDD Investment - {country}', fontsize=14)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('RDD Investment Value', fontsize=12)
    plt.legend(title='Investment Type', loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
snus.lineplot(
    data=selected_renewable_rdd, 
    x='TIME_PERIOD', 
    y='OBS_VALUE', 
    hue='Country/Region', 
    marker='o'
)
plt.title('RDD Investment Over Time by Country')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
def get_sum_gdp(country_list):
    return gdp_population_content[gdp_population_content['Country Name'].isin(country_list)].groupby('Country Name')['GDP'].sum()

def get_sum_population(country_list):
    return gdp_population_content[gdp_population_content['Country Name'].isin(country_list)].groupby('Country Name')['Population'].sum()